In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LoadJSON").master("local[*]").getOrCreate()

In [13]:
# Read a single line JSON file
df_single = spark.read.format("json").load("data/json/order_singleline.json")
df_single.printSchema()
df_single.show()

# it had identified the contact as an array with elements of long , customer as string, 
# order_id as string, order_line_items as an array of struct elements which holds amout double, item_id string, qty long

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
|[9000010002, 9000...|       C002|    O102|   [{25.5, I002, 1}]|
|        [9000010004]|       C001|    O103|[{51.2, I001, 3},...|
+--------------------+-----------+--------+--------------------+



In [14]:
# how to read a multi line JSON file
df_multi = spark.read.format("json").option("multiline", "true").load("data/json/order_multiline.json")
df_multi.printSchema()  
df_multi.show()

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
|[9000010002, 9000...|       C002|    O102|   [{25.5, I002, 1}]|
|        [9000010004]|       C001|    O103|[{51.2, I001, 3},...|
+--------------------+-----------+--------+--------------------+



In [15]:
# to read a JSON file as text and the content of the file will be stored in a single column named "value"
df = spark.read.format("text").load("data/json/order_singleline.json")
df.printSchema()
df.show(truncate=False)

root
 |-- value: string (nullable = true)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                 |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"order_id": "O101", "customer_id": "C001", "order_line_items": [{"item_id": "I001", "qty": 6, "amount": 102.45}, {"item_id": "I003", "qty": 2, "amount": 2.01}], "contact": [9000010000, 9000010001]}|
|{"order_id": "O102", "customer_id": "C002", "order_line_items": [{"item_id": "I002", "qty": 1, "amount": 25.50}], "contact": [9000010002, 9000010003]}  

In [16]:
# forcing spark to read JSON with custom schema
#schema name should match the column name in the JSON file, otherwise it will be null
_schema = "customer_id string, order_id string, contact array<long>"
df_schema = spark.read.format("json").schema(_schema).load("data/json/order_singleline.json")
df_schema.show()

+-----------+--------+--------------------+
|customer_id|order_id|             contact|
+-----------+--------+--------------------+
|       C001|    O101|[9000010000, 9000...|
|       C002|    O102|[9000010002, 9000...|
|       C001|    O103|        [9000010004]|
+-----------+--------+--------------------+



In [17]:
"""
root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
"""
_schema = "contact array<long>, customer_id string, order_id string, order_line_items array<struct<amount: double, item_id: string, qty: long>>"

df_schema_new = spark.read.format("json").schema(_schema).load("data/json/order_singleline.json")
df_schema_new.show()


+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
|[9000010002, 9000...|       C002|    O102|   [{25.5, I002, 1}]|
|        [9000010004]|       C001|    O103|[{51.2, I001, 3},...|
+--------------------+-----------+--------+--------------------+



In [ ]:
# Function from_json() to read a JSON 
_schema = "contact array<long>, customer_id string, order_id string, order_line_items array<struct<amount: double, item_id: string, qty: long>>"
df.show()

+--------------------+
|               value|
+--------------------+
|{"order_id": "O10...|
|{"order_id": "O10...|
|{"order_id": "O10...|
+--------------------+



In [ ]:
# from_json will take the string value and parse it into a struct type based on the schema provided.
from pyspark.sql.functions import from_json
df_expanded = df.withColumn("parsed", from_json(df.value, _schema))
df_expanded.printSchema()
df_expanded.show(truncate=False)

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+
|value                                                                                                                                                                                            

In [24]:
# to_json() will take a struct type and convert it into a JSON string
from pyspark.sql.functions import to_json

df_unparsed = df_expanded.withColumn("json", to_json(df_expanded.parsed))
df_unparsed.printSchema()
df_unparsed.show(truncate=False)

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)
 |-- json: string (nullable = true)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
df_expanded.printSchema()
df_expanded.show()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)

+--------------------+--------------------+
|               value|              parsed|
+--------------------+--------------------+
|{"order_id": "O10...|{[9000010000, 900...|
|{"order_id": "O10...|{[9000010002, 900...|
|{"order_id": "O10...|{[9000010004], C0...|
+--------------------+--------------------+



In [ ]:
# previously the parsed data was stored in a column named "parsed" which is of struct type,
# we can select the parsed data and expand it into multiple columns using the select() function and the "parsed.*" notation
df_1 = df_expanded.select("parsed.*")
df_1.printSchema()
df_1.show(truncate=False)

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
|[9000010002, 9000010003]|C002       |O102    |[{25.5, I002, 1}]                   |
|[9000010004]            |C001       |O103    |[{51.2, I001, 3}, {340.0, I004, 9}] |
+------------------------+-----------+--------+-----------------------------------

In [ ]:
# order_line_items  is an array of struct type, 
# we can use the explode() function to expand the array into multiple rows
# and drop the original array column using the drop() function
from pyspark.sql.functions import explode  
df_2 = df_1.withColumn("order_line_item", explode(df_1.order_line_items)).drop("order_line_items")
df_2.printSchema()
df_2.show(truncate=False)

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_item: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- item_id: string (nullable = true)
 |    |-- qty: long (nullable = true)

+------------------------+-----------+--------+-----------------+
|contact                 |customer_id|order_id|order_line_item  |
+------------------------+-----------+--------+-----------------+
|[9000010000, 9000010001]|C001       |O101    |{102.45, I001, 6}|
|[9000010000, 9000010001]|C001       |O101    |{2.01, I003, 2}  |
|[9000010002, 9000010003]|C002       |O102    |{25.5, I002, 1}  |
|[9000010004]            |C001       |O103    |{51.2, I001, 3}  |
|[9000010004]            |C001       |O103    |{340.0, I004, 9} |
+------------------------+-----------+--------+-----------------+



In [29]:
# now we can expand the struct type column "order_line_item" into multiple columns using the select() function and the "order_line_item.*" notation
df3 = df_2.select("customer_id", "order_id", "contact", "order_line_item.*")
df3.printSchema()
df3.show(truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- amount: double (nullable = true)
 |-- item_id: string (nullable = true)
 |-- qty: long (nullable = true)

+-----------+--------+------------------------+------+-------+---+
|customer_id|order_id|contact                 |amount|item_id|qty|
+-----------+--------+------------------------+------+-------+---+
|C001       |O101    |[9000010000, 9000010001]|102.45|I001   |6  |
|C001       |O101    |[9000010000, 9000010001]|2.01  |I003   |2  |
|C002       |O102    |[9000010002, 9000010003]|25.5  |I002   |1  |
|C001       |O103    |[9000010004]            |51.2  |I001   |3  |
|C001       |O103    |[9000010004]            |340.0 |I004   |9  |
+-----------+--------+------------------------+------+-------+---+



In [ ]:
# also we can expload the contact array into multiple rows using the explode() function
# expolading with the withColumn() function with same existing column name will replace the original column
df4 = df3.withColumn("contact", explode(df3.contact))
df4.printSchema()
df4.show(truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- contact: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- item_id: string (nullable = true)
 |-- qty: long (nullable = true)

+-----------+--------+----------+------+-------+---+
|customer_id|order_id|contact   |amount|item_id|qty|
+-----------+--------+----------+------+-------+---+
|C001       |O101    |9000010000|102.45|I001   |6  |
|C001       |O101    |9000010001|102.45|I001   |6  |
|C001       |O101    |9000010000|2.01  |I003   |2  |
|C001       |O101    |9000010001|2.01  |I003   |2  |
|C002       |O102    |9000010002|25.5  |I002   |1  |
|C002       |O102    |9000010003|25.5  |I002   |1  |
|C001       |O103    |9000010004|51.2  |I001   |3  |
|C001       |O103    |9000010004|340.0 |I004   |9  |
+-----------+--------+----------+------+-------+---+

